# Notebook 03 — Anomaly Detection
**Project:** Predictive Maintenance System — Microsoft Azure Dataset  
**Author:** Abrar Ahmed  
**Phase:** 2 — Detect abnormal sensor behaviour

---
### What we build
Two complementary anomaly detection models:

| Model | Type | Strength |
|---|---|---|
| Isolation Forest | Unsupervised, tree-based | Fast, interpretable, strong baseline |
| LSTM Autoencoder | Unsupervised, deep learning | Captures temporal patterns in sequences |

Both are **unsupervised** — they learn what *normal* looks like, then flag deviations.  
This is realistic: in production you rarely have perfectly labelled failure data.

### Key concept: anomaly ≠ failure
An anomaly is unusual sensor behaviour. A failure is a confirmed breakdown.  
Anomalies are *leading indicators* — they appear before failures.  
Our goal: detect anomalies early enough to schedule preventive maintenance.

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score, f1_score
)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import mlflow
import mlflow.pytorch
import mlflow.sklearn

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PROCESSED = Path('../data/processed')
MODELS_DIR     = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch version : {torch.__version__}')
print(f'Device        : {device}')
print('Imports OK')

## 1. Load features and prepare data

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / 'features.parquet')
df = df.sort_values(['machineID', 'datetime']).reset_index(drop=True)

print(f'Loaded: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nLabel distribution:')
print(df['failure_label'].value_counts())

In [ ]:
# Define feature columns for anomaly detection
# We use: raw sensor readings + rolling stats (mean, std)
# Rolling std is particularly powerful — it captures volatility increases before failure

raw_sensors   = ['volt', 'rotate', 'pressure', 'vibration']
rolling_means = [c for c in df.columns if '_mean_' in c]
rolling_stds  = [c for c in df.columns if '_std_'  in c]
error_feats   = [c for c in df.columns if 'error_' in c and '_24h' in c]
time_feats    = ['hour_of_day', 'day_of_week', 'is_weekend']
maint_feats   = ['hours_since_maint']

FEATURE_COLS = raw_sensors + rolling_means + rolling_stds + error_feats + time_feats + maint_feats
# Remove any that don't exist in the dataframe
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print(f'Features for anomaly detection: {len(FEATURE_COLS)}')
print(f'  Raw sensors    : {len(raw_sensors)}')
print(f'  Rolling means  : {len(rolling_means)}')
print(f'  Rolling stds   : {len(rolling_stds)}')
print(f'  Error features : {len(error_feats)}')
print(f'  Time features  : {len(time_feats)}')
print(f'  Maint features : {len(maint_feats)}')

In [ ]:
# Split into train (normal only) and test (all data)
# IMPORTANT: anomaly detection trains on NORMAL data only
# The model learns what normal looks like, then flags deviations
# We use the first 8 months for training, last 4 for testing

cutoff_date = pd.Timestamp('2015-09-01')

train_df = df[df['datetime'] < cutoff_date].copy()
test_df  = df[df['datetime'] >= cutoff_date].copy()

# Training set: normal rows only (unsupervised — we pretend we don't know labels)
train_normal = train_df[train_df['failure_label'] == 0]

print(f'Train set (normal only) : {len(train_normal):>8,} rows')
print(f'Test set  (all)         : {len(test_df):>8,} rows')
print(f'  Test positives        : {test_df["failure_label"].sum():>8,}')
print(f'  Test negatives        : {(test_df["failure_label"]==0).sum():>8,}')

In [ ]:
# Scale features — critical for both models
# IsolationForest is tree-based so less sensitive to scale,
# but LSTM Autoencoder needs scaled inputs to train stably

scaler = StandardScaler()
X_train = scaler.fit_transform(train_normal[FEATURE_COLS].fillna(0))
X_test  = scaler.transform(test_df[FEATURE_COLS].fillna(0))
y_test  = test_df['failure_label'].values

print(f'X_train shape : {X_train.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'y_test shape  : {y_test.shape}')
print(f'Scaler mean (first 4): {scaler.mean_[:4].round(2)}')

## 2. Model 1 — Isolation Forest

### How it works
Isolation Forest builds random decision trees. Each tree tries to *isolate* a data point  
by randomly splitting features. Anomalies are isolated faster (fewer splits needed)  
because they sit far from the bulk of normal data.

**Score:** Average path length across all trees. Short path = anomaly.

**Why use it:**  
- No labels needed  
- Handles high-dimensional data well  
- Fast to train  
- Interpretable intuition

In [ ]:
# Start MLflow experiment
# MLflow tracks every run: params, metrics, model artefacts
# This is what CERN's JD calls 'experiment tracking and ML lifecycle management'

mlflow.set_experiment('anomaly_detection')
print('MLflow experiment set: anomaly_detection')
print('Run: mlflow ui --host 0.0.0.0 in terminal to view the dashboard')

In [ ]:
# Train Isolation Forest
# contamination: expected proportion of anomalies in training data
# We set it to ~2% matching our label distribution
# n_estimators: number of trees — 200 gives stable results

with mlflow.start_run(run_name='isolation_forest_v1'):

    # --- Parameters ---
    params = {
        'n_estimators'  : 200,
        'contamination' : 0.02,
        'max_samples'   : 'auto',
        'random_state'  : SEED,
        'n_jobs'        : -1,       # use all CPU cores
    }
    mlflow.log_params(params)

    # --- Train ---
    print('Training Isolation Forest...')
    iso_forest = IsolationForest(**params)
    iso_forest.fit(X_train)
    print('Training complete.')

    # --- Score ---
    # score_samples returns negative anomaly scores
    # More negative = more anomalous
    # We negate so that higher = more anomalous (easier to threshold)
    iso_scores = -iso_forest.score_samples(X_test)

    # --- Threshold at 98th percentile (top 2% flagged as anomalies) ---
    threshold = np.percentile(iso_scores, 98)
    iso_preds = (iso_scores >= threshold).astype(int)

    # --- Metrics ---
    f1  = f1_score(y_test, iso_preds, zero_division=0)
    auc = roc_auc_score(y_test, iso_scores)

    mlflow.log_metric('f1_score',  f1)
    mlflow.log_metric('roc_auc',   auc)
    mlflow.log_metric('threshold', threshold)

    # --- Log model ---
    mlflow.sklearn.log_model(iso_forest, 'isolation_forest')

    print(f'\n=== ISOLATION FOREST RESULTS ===')
    print(f'Threshold (98th pct) : {threshold:.4f}')
    print(f'F1 Score             : {f1:.4f}')
    print(f'ROC-AUC              : {auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, iso_preds, target_names=['normal', 'anomaly']))

In [ ]:
# Precision-Recall curve — more informative than ROC for imbalanced data
# We want HIGH recall (catch most failures) at acceptable precision (not too many false alarms)

precision, recall, thresholds = precision_recall_curve(y_test, iso_scores)

# Find threshold that maximises F1
f1_scores_curve = 2 * precision * recall / (precision + recall + 1e-8)
best_idx        = np.argmax(f1_scores_curve)
best_threshold  = thresholds[best_idx]
best_f1         = f1_scores_curve[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: PR curve
axes[0].plot(recall, precision, color='#185FA5', linewidth=2)
axes[0].scatter(recall[best_idx], precision[best_idx],
                color='red', s=100, zorder=5,
                label=f'Best F1={best_f1:.3f} @ threshold={best_threshold:.3f}')
axes[0].set_xlabel('Recall (fraction of failures caught)')
axes[0].set_ylabel('Precision (fraction of alarms correct)')
axes[0].set_title('Isolation Forest — Precision-Recall Curve', fontweight='bold')
axes[0].legend(fontsize=9)

# Right: anomaly score distribution
axes[1].hist(iso_scores[y_test == 0], bins=80, alpha=0.6,
             color='#185FA5', density=True, label='normal', edgecolor='none')
axes[1].hist(iso_scores[y_test == 1], bins=80, alpha=0.7,
             color='#A32D2D', density=True, label='pre-failure', edgecolor='none')
axes[1].axvline(threshold, color='black', linestyle='--', linewidth=1.5,
                label=f'threshold={threshold:.3f}')
axes[1].set_xlabel('Anomaly score (higher = more anomalous)')
axes[1].set_ylabel('Density')
axes[1].set_title('Score distributions: normal vs pre-failure', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'iso_forest_results.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved — use this in your README')

In [ ]:
# Re-evaluate with the optimal threshold
iso_preds_opt = (iso_scores >= best_threshold).astype(int)
f1_opt = f1_score(y_test, iso_preds_opt, zero_division=0)

print(f'=== ISOLATION FOREST (optimal threshold) ===')
print(f'Threshold : {best_threshold:.4f}')
print(f'F1 Score  : {f1_opt:.4f}')
print()
print(classification_report(y_test, iso_preds_opt, target_names=['normal', 'anomaly']))

# Save the optimal threshold for later use in the API
iso_threshold = best_threshold

## 3. Model 2 — LSTM Autoencoder

### How it works
An autoencoder is a neural network trained to reconstruct its input.  
- **Encoder**: compresses the input sequence into a small latent representation  
- **Decoder**: reconstructs the original sequence from the latent representation  
- **Training**: minimise reconstruction error on NORMAL data only  
- **Inference**: high reconstruction error = the input looks different from normal = anomaly  

The LSTM part means the model processes sequences of timesteps,  
capturing temporal dependencies — crucial for sensor data.

```
Input sequence        Encoder          Latent        Decoder         Reconstruction
[t-11, ..., t-1, t]  → LSTM → ... →  [z]  → ... →  LSTM →  [t-11, ..., t-1, t]
```
**Anomaly score** = mean squared error between input and reconstruction.

In [ ]:
SEQ_LEN = 6  # 6 hours of context per sample

def build_sequences(data_array, seq_len):
    sequences = []
    for i in range(len(data_array) - seq_len + 1):
        sequences.append(data_array[i : i + seq_len])
    return np.array(sequences)

def build_sequences_per_machine(df_subset, feature_cols, seq_len):
    all_seqs   = []
    all_labels = []
    for machine_id, group in df_subset.groupby('machineID'):
        group   = group.sort_values('datetime')
        X_group = scaler.transform(group[feature_cols].fillna(0))
        y_group = group['failure_label'].values
        seqs    = build_sequences(X_group, seq_len)
        labels  = y_group[seq_len - 1:]
        all_seqs.append(seqs)
        all_labels.append(labels)
    return np.concatenate(all_seqs), np.concatenate(all_labels)

print('Building training sequences (normal only)...')
X_seq_train, y_seq_train = build_sequences_per_machine(
    train_normal, FEATURE_COLS, SEQ_LEN
)

print('Building test sequences...')
X_seq_test, y_seq_test = build_sequences_per_machine(
    test_df, FEATURE_COLS, SEQ_LEN
)

print(f'Train sequences : {X_seq_train.shape}  (samples x seq_len x features)')
print(f'Test sequences  : {X_seq_test.shape}')
print(f'Test positives  : {y_seq_test.sum():,} ({100*y_seq_test.mean():.1f}%)')

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, latent_dim=8, num_layers=1, dropout=0.0):
        super().__init__()
        self.seq_len    = SEQ_LEN
        self.latent_dim = latent_dim

        self.encoder_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )
        self.encoder_fc = nn.Linear(hidden_dim, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )
        self.output_fc = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, (h_n, _) = self.encoder_lstm(x)
        latent       = self.encoder_fc(h_n[-1])
        dec_input    = self.decoder_fc(latent)
        dec_input    = dec_input.unsqueeze(1).repeat(1, self.seq_len, 1)
        dec_out, _   = self.decoder_lstm(dec_input)
        return self.output_fc(dec_out)

    def reconstruction_error(self, x):
        recon = self.forward(x)
        return torch.mean((x - recon) ** 2, dim=(1, 2))


INPUT_DIM = X_seq_train.shape[2]
model = LSTMAutoencoder(input_dim=INPUT_DIM).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input dim       : {INPUT_DIM}')
print(f'Trainable params: {n_params:,}')

In [ ]:
BATCH_SIZE = 512
EPOCHS     = 10
LR         = 1e-3

X_train_tensor = torch.FloatTensor(X_seq_train).to(device)
X_test_tensor  = torch.FloatTensor(X_seq_test).to(device)

train_dataset = TensorDataset(X_train_tensor, X_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.MSELoss()

train_losses  = []
batch_size_eval = 1024

print(f'Batches per epoch : {len(train_loader)}')
print(f'Training for {EPOCHS} epochs...')
print(f'Epoch | Train Loss')
print('-' * 25)

with mlflow.start_run(run_name='lstm_autoencoder_v1'):

    mlflow.log_params({
        'model_type' : 'LSTMAutoencoder',
        'seq_len'    : SEQ_LEN,
        'hidden_dim' : 32,
        'latent_dim' : 8,
        'num_layers' : 1,
        'batch_size' : BATCH_SIZE,
        'epochs'     : EPOCHS,
        'lr'         : LR,
        'input_dim'  : INPUT_DIM,
    })

    model.train()
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0.0
        for X_batch, _ in train_loader:
            optimizer.zero_grad()
            recon = model(X_batch)
            loss  = criterion(recon, X_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        scheduler.step(avg_loss)
        mlflow.log_metric('train_loss', avg_loss, step=epoch)
        print(f'  {epoch:>3}   |   {avg_loss:.6f}')

    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        recon_errors = []
        for i in range(0, len(X_test_tensor), batch_size_eval):
            batch = X_test_tensor[i:i+batch_size_eval]
            recon_errors.append(model.reconstruction_error(batch).cpu().numpy())
        recon_errors = np.concatenate(recon_errors)

        train_errors_list = []
        for i in range(0, len(X_train_tensor), batch_size_eval):
            batch = X_train_tensor[i:i+batch_size_eval]
            train_errors_list.append(model.reconstruction_error(batch).cpu().numpy())
        train_errors = np.concatenate(train_errors_list)

    lstm_threshold = np.percentile(train_errors, 98)
    lstm_preds     = (recon_errors >= lstm_threshold).astype(int)
    f1_lstm        = f1_score(y_seq_test, lstm_preds, zero_division=0)
    auc_lstm       = roc_auc_score(y_seq_test, recon_errors)

    mlflow.log_metric('f1_score',  f1_lstm)
    mlflow.log_metric('roc_auc',   auc_lstm)
    mlflow.log_metric('threshold', lstm_threshold)
    mlflow.pytorch.log_model(model, 'lstm_autoencoder')

    print(f'\n=== LSTM AUTOENCODER RESULTS ===')
    print(f'Threshold (98th) : {lstm_threshold:.6f}')
    print(f'F1 Score         : {f1_lstm:.4f}')
    print(f'ROC-AUC          : {auc_lstm:.4f}')
    print(classification_report(y_seq_test, lstm_preds, target_names=['normal', 'anomaly']))

In [ ]:
# Training loss curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: loss curve
axes[0].plot(range(1, EPOCHS + 1), train_losses, color='#185FA5', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('LSTM Autoencoder — Training Loss', fontweight='bold')

# Right: reconstruction error distributions
axes[1].hist(recon_errors[y_seq_test == 0], bins=80, alpha=0.6,
             color='#185FA5', density=True, label='normal', edgecolor='none')
axes[1].hist(recon_errors[y_seq_test == 1], bins=80, alpha=0.7,
             color='#A32D2D', density=True, label='pre-failure', edgecolor='none')
axes[1].axvline(lstm_threshold, color='black', linestyle='--', linewidth=1.5,
                label=f'threshold={lstm_threshold:.4f}')
axes[1].set_xlabel('Reconstruction error (higher = more anomalous)')
axes[1].set_ylabel('Density')
axes[1].set_title('LSTM AE — Reconstruction error distributions', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'lstm_ae_results.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Visualise anomalies on a machine timeline

This is the plot that tells the story. We pick one machine and show:  
- Its sensor readings over time  
- Where Isolation Forest flagged anomalies  
- Where actual failures occurred  

A good model flags anomalies BEFORE the red failure line.

In [ ]:
# Add anomaly scores back to the test dataframe
test_with_scores = test_df.copy().reset_index(drop=True)

# Isolation Forest scores (one per row in test_df)
test_with_scores['iso_score']     = iso_scores
test_with_scores['iso_anomaly']   = iso_preds_opt

# Load failures for overlay
failures_raw = pd.read_parquet(DATA_PROCESSED / 'failures.parquet')

# Pick a machine in test set that has at least one failure
test_failures = failures_raw[
    failures_raw['datetime'] >= pd.Timestamp('2015-09-01')
]
machines_with_test_failures = test_failures['machineID'].value_counts()
target_machine = machines_with_test_failures.index[0]

m_test = test_with_scores[test_with_scores['machineID'] == target_machine]
m_fail = test_failures[test_failures['machineID'] == target_machine]

print(f'Visualising machine {target_machine}')
print(f'Test rows : {len(m_test):,}')
print(f'Failures  : {len(m_fail)}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# Top: volt sensor
axes[0].plot(m_test['datetime'], m_test['volt'],
             color='#185FA5', linewidth=0.7, alpha=0.8, label='volt')
axes[0].set_ylabel('Volt', fontweight='bold')

# Middle: vibration sensor
axes[1].plot(m_test['datetime'], m_test['vibration'],
             color='#BA7517', linewidth=0.7, alpha=0.8, label='vibration')
axes[1].set_ylabel('Vibration', fontweight='bold')

# Bottom: anomaly score
axes[2].plot(m_test['datetime'], m_test['iso_score'],
             color='#5DCAA5', linewidth=0.7, alpha=0.8, label='anomaly score')
axes[2].axhline(best_threshold, color='orange', linestyle='--',
                linewidth=1.5, label=f'threshold={best_threshold:.3f}')

# Shade anomaly regions
anomaly_mask = m_test['iso_anomaly'] == 1
axes[2].fill_between(m_test['datetime'], 0, m_test['iso_score'],
                     where=anomaly_mask, alpha=0.3, color='red', label='flagged anomaly')
axes[2].set_ylabel('Anomaly Score', fontweight='bold')
axes[2].legend(loc='upper left', fontsize=8)

# Overlay failure events on all panels
for _, fail_row in m_fail.iterrows():
    for ax in axes:
        ax.axvline(fail_row['datetime'], color='red',
                   linewidth=2, linestyle='--', alpha=0.9)

# Legend for failure lines
from matplotlib.lines import Line2D
axes[0].legend(
    handles=[Line2D([0],[0], color='red', linestyle='--', linewidth=2, label='actual failure')],
    loc='upper right', fontsize=9
)

axes[0].set_title(
    f'Machine {target_machine} — Sensors and Isolation Forest anomaly detection',
    fontweight='bold', fontsize=12
)
axes[-1].set_xlabel('Date')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'anomaly_timeline.png', bbox_inches='tight', dpi=150)
plt.show()
print('This is your key README plot — anomalies flagged BEFORE the failure lines.')

## 5. Model comparison

In [ ]:
# Compare both models on common test set
# Note: LSTM uses sequences (slightly fewer samples) so we align on shared index

print('=== MODEL COMPARISON ===')
print(f'{"Model":<25} {"F1":>8} {"ROC-AUC":>10}')
print('-' * 45)

# Isolation Forest (on full test set)
f1_iso  = f1_score(y_test, iso_preds_opt, zero_division=0)
auc_iso = roc_auc_score(y_test, iso_scores)
print(f'{"Isolation Forest":<25} {f1_iso:>8.4f} {auc_iso:>10.4f}')

# LSTM Autoencoder (on sequence test set)
print(f'{"LSTM Autoencoder":<25} {f1_lstm:>8.4f} {auc_lstm:>10.4f}')

print()
print('Note: LSTM uses sequences (12h windows) so test set is slightly smaller.')
print('Both models are unsupervised — no failure labels used during training.')

## 6. Save models and artefacts

In [ ]:
import joblib

# Save Isolation Forest + scaler
joblib.dump(iso_forest, MODELS_DIR / 'iso_forest.joblib')
joblib.dump(scaler,     MODELS_DIR / 'scaler.joblib')

# Save LSTM Autoencoder
torch.save({
    'model_state_dict' : model.state_dict(),
    'input_dim'        : INPUT_DIM,
    'seq_len'          : SEQ_LEN,
    'hidden_dim'       : 64,
    'latent_dim'       : 16,
    'num_layers'       : 2,
    'iso_threshold'    : iso_threshold,
    'lstm_threshold'   : lstm_threshold,
    'feature_cols'     : FEATURE_COLS,
}, MODELS_DIR / 'lstm_autoencoder.pt')

# Save feature columns list (needed by the API)
import json
with open(MODELS_DIR / 'feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('Models saved to models/')
print('  iso_forest.joblib')
print('  scaler.joblib')
print('  lstm_autoencoder.pt')
print('  feature_cols.json')
print()
print('=== NOTEBOOK 03 COMPLETE ===')
print('Next: open 04_failure_pred.ipynb')